In [1]:
import numpy as np  # linear algebra
import pandas as pd  # data processing, CSV file I/O (e.g. pd.read_csv)
import os

for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        print(os.path.join(dirname, filename))




/kaggle/input/train.csv
/kaggle/input/description.md
/kaggle/input/sample_submission.csv.zip
/kaggle/input/test.csv.zip
/kaggle/input/train.csv.zip
/kaggle/input/test.csv
/kaggle/input/sample_submission.csv
/kaggle/input/playground-series-s3e18/train.csv
/kaggle/input/playground-series-s3e18/description.md
/kaggle/input/playground-series-s3e18/sample_submission.csv.zip
/kaggle/input/playground-series-s3e18/test.csv.zip
/kaggle/input/playground-series-s3e18/train.csv.zip
/kaggle/input/playground-series-s3e18/test.csv
/kaggle/input/playground-series-s3e18/sample_submission.csv


In [2]:
from matplotlib import pyplot as plt
import seaborn as sns

from scipy import stats
from scipy.stats import norm, skew, boxcox
from collections import Counter

# The sweetviz package is optional for EDA and can be omitted to keep the runtime light.




In [3]:
# Load data
sample_submission = pd.read_csv(
    "/kaggle/input/playground-series-s3e18/sample_submission.csv"
)
train = pd.read_csv("/kaggle/input/playground-series-s3e18/train.csv")
test = pd.read_csv("/kaggle/input/playground-series-s3e18/test.csv")




In [4]:
# Quick sanity check
train.head()




,id,BertzCT,Chi1,Chi1n,Chi1v,Chi2n,Chi2v,Chi3v,Chi4n,EState_VSA1,...,SlogP_VSA3,VSA_EState9,fr_COO,fr_COO2,EC1,EC2,EC3,EC4,EC5,EC6
0,3305,212.928441,6.270857,3.431698,4.045502,1.811731,2.272429,1.619489,0.828835,7.822697,...,13.825658,36.819857,0,0,1,1,0,1,0,0
1,8217,221.694355,4.060660,3.548618,3.548618,2.499652,2.499652,1.607504,0.883813,17.980451,...,9.589074,39.833333,2,2,1,0,0,1,0,0
2,12616,159.130752,4.414719,2.801636,2.801636,2.676274,2.676274,1.204209,0.361282,12.011146,...,11.215359,33.500000,1,1,1,1,0,0,0,0
3,10153,1239.282494,16.098662,10.151117,10.151117,11.008637,11.008637,7.914542,3.408511,5.969305,...,9.589074,73.500000,1,1,1,1,0,0,0,0
4,9564,574.170559,8.417121,5.143525,5.143525,3.478542,3.478542,2.264463,2.087653,5.969305,...,4.794537,39.166667,1,1,1,0,0,1,0,0


In [5]:
# Inspect dtypes (optional)
train.dtypes




id                     int64
BertzCT              float64
Chi1                 float64
Chi1n                float64
Chi1v                float64
Chi2n                float64
Chi2v                float64
Chi3v                float64
Chi4n                float64
EState_VSA1          float64
EState_VSA2          float64
ExactMolWt           float64
FpDensityMorgan1     float64
FpDensityMorgan2     float64
FpDensityMorgan3     float64
HallKierAlpha        float64
HeavyAtomMolWt       float64
Kappa3               float64
MaxAbsEStateIndex    float64
MinEStateIndex       float64
NumHeteroatoms         int64
PEOE_VSA10           float64
PEOE_VSA14           float64
PEOE_VSA6            float64
PEOE_VSA7            float64
PEOE_VSA8            float64
SMR_VSA10            float64
SMR_VSA5             float64
SlogP_VSA3           float64
VSA_EState9          float64
fr_COO                 int64
fr_COO2                int64
EC1                    int64
EC2                    int64
EC3           

In [6]:
# (Optional) generate a Sweetviz report if the package is available.
# import sweetviz as sv
# my_report = sv.analyze(train)
# my_report.show_html()  # Generates "SWEETVIZ_REPORT.html"




In [7]:
# -----------------------------------------
# Model training & validation (lightweight RandomForest)
# -----------------------------------------
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# Define feature columns (exclude id and all EC target columns)
target_cols = ["EC1", "EC2", "EC3", "EC4", "EC5", "EC6"]
feature_cols = [c for c in train.columns if c not in target_cols + ["id"]]

X = train[feature_cols]
y_ec1 = train["EC1"]
y_ec2 = train["EC2"]

# Simple train/validation split to monitor local AUC
X_train, X_val, y_ec1_train, y_ec1_val = train_test_split(
    X, y_ec1, test_size=0.2, random_state=42, stratify=y_ec1
)
_, _, y_ec2_train, y_ec2_val = train_test_split(
    X, y_ec2, test_size=0.2, random_state=42, stratify=y_ec2
)

# Shared RandomForest parameters
rf_params = {
    "n_estimators": 200,
    "max_depth": None,
    "min_samples_split": 2,
    "min_samples_leaf": 1,
    "random_state": 42,
    "n_jobs": 5,
    "class_weight": "balanced",
}

# Train model for EC1
rf_ec1 = RandomForestClassifier(**rf_params)
rf_ec1.fit(X_train, y_ec1_train)
val_pred_ec1 = rf_ec1.predict_proba(X_val)[:, 1]
auc_ec1 = roc_auc_score(y_ec1_val, val_pred_ec1)
print(f"Local validation AUC for EC1: {auc_ec1:.5f}")

# Train model for EC2
rf_ec2 = RandomForestClassifier(**rf_params)
rf_ec2.fit(X_train, y_ec2_train)
val_pred_ec2 = rf_ec2.predict_proba(X_val)[:, 1]
auc_ec2 = roc_auc_score(y_ec2_val, val_pred_ec2)
print(f"Local validation AUC for EC2: {auc_ec2:.5f}")




Local validation AUC for EC1: 0.69541
Local validation AUC for EC2: 0.47123


In [8]:
# -----------------------------------------
# Predict on test set and write submission file
# -----------------------------------------
test_features = test[feature_cols]

test_pred_ec1 = rf_ec1.predict_proba(test_features)[:, 1]
test_pred_ec2 = rf_ec2.predict_proba(test_features)[:, 1]

submission = pd.DataFrame(
    {"id": test["id"], "EC1": test_pred_ec1, "EC2": test_pred_ec2}
)

# Write to the working directory so Kaggle can pick it up
output_path = "/kaggle/working/submission.csv"
submission.to_csv(output_path, index=False)
print(f"Submission file written to {output_path}")

Submission file written to /kaggle/working/submission.csv
